# CV03 — XGBoost Model (Machine Learning)
**Thành viên:** Trần Thị Thanh Phương  
**Deadline:** 08/11/2026

Notebook thực nghiệm cho `src/cv03_xgboost_model.py`.

In [ ]:
import sys

sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from xgboost import XGBRegressor

PROJECT_ROOT = Path('..').resolve()
DATA_PROC = PROJECT_ROOT / 'data' / 'processed'

## Step 1: Build Flat Feature Matrix (96 lags + rolling)

In [ ]:
import sys
import pandas as pd

src_dir = "/content/QLCTG_nhom7/time-series-lab4/src"
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from cv03_xgboost_model import (
    step1_build_flat_features, LAG, PROCESSED_DIR
)

train = pd.read_csv(
    PROCESSED_DIR / "train.csv",
    index_col=0,
    parse_dates=True
)
val = pd.read_csv(
    PROCESSED_DIR / "val.csv",
    index_col=0,
    parse_dates=True
)
test = pd.read_csv(
    PROCESSED_DIR / "test.csv",
    index_col=0,
    parse_dates=True
)

# Ghép dữ liệu theo thời gian để giữ lịch sử cho Val/Test
all_data = pd.concat([train, val, test])
all_features = step1_build_flat_features(all_data, lag=LAG)

train_features = all_features.loc[train.index].dropna()
val_features = all_features.loc[val.index]
test_features = all_features.loc[test.index]

feature_cols = [
    c for c in train_features.columns if c != "load_t0"
]

X_train = train_features[feature_cols]
y_train = train_features["load_t0"]

X_val = val_features[feature_cols]
y_val = val_features["load_t0"]

X_test = test_features[feature_cols]
y_test = test_features["load_t0"]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)


## Step 2: Train XGBoost with Early Stopping

In [ ]:
from cv03_xgboost_model import (
    step2_train_xgboost, MODELS_DIR
)

# Huấn luyện trên Train, dùng Validation để theo dõi
model, evals_result = step2_train_xgboost(
    X_train, y_train, X_val, y_val
)

# Lưu mô hình đã huấn luyện
model.save_model("/content/cv03_xgboost_test.json")

print("✅ Huấn luyện XGBoost thành công!")
print("Best iteration:", model.best_iteration)


## Step 3: Evaluate & Feature Importance

In [ ]:
import pickle
import pandas as pd

from cv03_xgboost_model import (
    step3_evaluate,
    step4_feature_importance,
    PROCESSED_DIR,
    TABLES_DIR
)

# Đọc scaler đã sử dụng khi tiền xử lý
with open(PROCESSED_DIR / "scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

# Đánh giá mô hình trên tập Test
metrics, y_true, y_pred = step3_evaluate(
    model, X_test, y_test, scaler
)

metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(
    TABLES_DIR / "cv03_xgb_metrics.csv",
    index=False
)

print("=== KẾT QUẢ ĐÁNH GIÁ ===")
print(metrics_df.to_string(index=False))

# Phân tích Feature Importance theo Gain
fi_df = step4_feature_importance(
    model, feature_cols
)

print("\n=== TOP 10 FEATURES ===")
print(fi_df.head(10).to_string(index=False))
